# Newton's method check (Rust)

Start JupyterLab with `workshop run -- notebook`. Uses the `thole` facade crate from the working tree. After changing the Rust code, run the `:dep` cell again.

In [ ]:
:dep thole = { path = "/project/crates/thole" }

In [ ]:
use thole::core::Tolerance;
use thole::roots::{brent, newton, Error, Options};

## 1. A first call

The Python binding runs this same code, so the bits must equal the `hex` printed by the Python notebook.

In [ ]:
let r = newton(|x| x * x - 2.0, |x| 2.0 * x, 1.0, &Options::default()).unwrap();
println!("{r:?}");
println!("error = {:e}", (r.x - 2.0_f64.sqrt()).abs());
println!("bits  = {:#018x}", r.x.to_bits());

## 2. Reference roots (mpmath, 50 digits), next to Brent's method

In [ ]:
let opts = Options::default();
let cases: [(&str, fn(f64) -> f64, fn(f64) -> f64, f64, f64, f64); 3] = [
    ("x^3 - x - 2", |x| x * x * x - x - 2.0, |x| 3.0 * x * x - 1.0, 1.5, 2.0, 1.5213797068045676),
    ("cos x - x", |x| x.cos() - x, |x| -x.sin() - 1.0, 0.5, 1.0, 0.7390851332151607),
    ("x e^x - 1", |x| x * x.exp() - 1.0, |x| (x + 1.0) * x.exp(), 0.5, 1.0, 0.5671432904097838),
];
for (name, f, df, x0, b, expected) in cases {
    let r = newton(f, df, x0, &opts).unwrap();
    let by_brent = brent(f, 0.0, b, &opts).unwrap();
    let err = (r.x - expected).abs();
    println!("{name:12} x={:.16} err={err:.3e} steps={} (brent: {}) ok={}", r.x, r.iterations, by_brent.iterations, err <= opts.tol.bound(r.x));
}

## 3. What the Lean theorems predict

- `newtonStep_error`: for `x^2 - 2`, the error after a step is at most `e^2 / x`.
- `newton_error_of_done`: close to a simple root, the error is at most the last step.

In [ ]:
let root = 2.0_f64.sqrt();
let mut x: f64 = 3.0;
for _ in 0..6 {
    let e = (x - root).abs();
    let next = x - (x * x - 2.0) / (2.0 * x);
    println!("x={next:.16} error={:.3e} bound={:.3e}", (next - root).abs(), e * e / x);
    assert!((next - root).abs() <= e * e / x + 4.0 * f64::EPSILON);
    x = next;
}
for xtol in [0.5, 0.1, 1e-2, 1e-4, 1e-8] {
    let opts = Options { tol: Tolerance::new(xtol, 0.0), max_iter: 100 };
    let r = newton(|x| x * x - 2.0, |x| 2.0 * x, 2.0, &opts).unwrap();
    println!("xtol={xtol:<6e} x={:.16} error={:.3e} last step={:.3e} steps={}", r.x, (r.x - root).abs(), r.step.abs(), r.iterations);
    assert!((r.x - root).abs() <= r.step.abs() && r.step.abs() <= xtol);
}

## 4. Failures

In [ ]:
let opts = Options::default();
let results: Vec<(&str, Result<_, Error>)> = vec![
    ("cycle", newton(|x| x * x * x - 2.0 * x + 2.0, |x| 3.0 * x * x - 2.0, 0.0, &opts)),
    ("run-away", newton(f64::atan, |x| 1.0 / (1.0 + x * x), 10.0, &opts)),
    ("zero slope", newton(|x| x * x - 2.0, |x| 2.0 * x, 0.0, &opts)),
    ("infinite start", newton(|x| x, |_| 1.0, f64::INFINITY, &opts)),
    ("f returns NaN", newton(|_| f64::NAN, |_| 1.0, 1.0, &opts)),
    ("step overflows", newton(|_| 1.0, |_| 1e-310, 0.0, &opts)),
];
for (name, r) in &results {
    match r {
        Ok(root) => println!("{name:16} -> Ok({root:?})"),
        Err(e) => println!("{name:16} -> {e}   [{e:?}]"),
    }
}